# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane lock: I confirm Lane 4, CTR / Engagement Opportunity Scoring.** Data: the anonymized starter slice (`data/raw/content_refresh_anonymized.csv`, trailing 90 days). The decision moment is *today*, the end of that window, so the queue uses only what has already been observed. Rates are on the ×100 scale (`0.49` = 0.49%).

## 1. My rule and its reason codes

**Signal checks first.** Before I trust a rule, I check the two signals it leans on. Both are linked to real FlyRank flags from the session:

- **Signal A, CTR vs position** (behind the CTR-fix logic). Claim: CTR falls as position gets worse, so a page should be compared only with pages at a similar position.
- **Signal B, staleness** (behind the refresh flags). Claim: pages not updated for longer under-capture clicks more often.

Population: pages with known position (`avg_position > 0`) and at least 1,000 impressions in 90 days, so CTR isn't built on a handful of views.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/content_refresh_anonymized.csv").exists())
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")

MIN_IMPR = 1_000
pop = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= MIN_IMPR)].copy()

# Signal A: CTR by position bucket
POS_BINS = [0, 3, 5, 10, 15, 20, 50, 1000]
POS_LABELS = ["1-3", "3-5", "5-10", "10-15", "15-20", "20-50", "50+"]
pop["pos_bucket"] = pd.cut(pop["avg_position"], bins=POS_BINS, labels=POS_LABELS)
sig_a = pop.groupby("pos_bucket", observed=True).agg(
    n=("ctr", "size"), impressions=("impressions_90d", "sum"), clicks=("clicks_90d", "sum"), median_ctr=("ctr", "median"))
sig_a["pooled_ctr_%"] = 100 * sig_a["clicks"] / sig_a["impressions"]
print("Signal A: CTR vs position")
print(sig_a[["n", "pooled_ctr_%", "median_ctr"]].round(3))

Signal A: CTR vs position
               n  pooled_ctr_%  median_ctr
pos_bucket                                
1-3          416         0.491       0.235
3-5         1539         0.456       0.330
5-10        4636         0.292       0.220
10-15       2076         0.362       0.190
15-20       1344         0.346       0.180
20-50       3312         0.155       0.090
50+          189         0.036       0.000


**Verdict A: MIXED.** The overall direction holds: positions 1–5 earn about 0.46–0.49% pooled CTR, positions 20–50 earn 0.16%, and 50+ earn 0.04%. But it isn't monotonic. Positions 5–10 (0.29%) earn *less* than 10–20 (0.35–0.36%). So the coarse FlyRank-style tiers (`top_3` / `page_1` / `striking`) would give a wrong benchmark. **What this changes in my rule:** expected CTR is computed per narrow position bucket, measured from the data, not assumed from a textbook curve.

In [2]:
# Signal B: staleness vs. under-capture, among visible pages (positions 1-20)
vis = pop[pop["avg_position"] <= 20].copy()
bucket_ctr = sig_a["pooled_ctr_%"]
vis["expected_ctr"] = vis["pos_bucket"].map(bucket_ctr).astype(float)
vis["under_capture"] = vis["ctr"] < 0.5 * vis["expected_ctr"]

FRESH_ORDER = ["0-30", "31-90", "91-180", "181+"]
sig_b = vis.groupby("freshness_tier").agg(n=("ctr", "size"), under_capture_share=("under_capture", "mean")).reindex(FRESH_ORDER)
print("Signal B: days since last update vs. share of pages under half their position's CTR")
print(sig_b.round(3))

Signal B: days since last update vs. share of pages under half their position's CTR
                   n  under_capture_share
freshness_tier                           
0-30            6299                0.374
31-90             38                0.500
91-180          3668                0.416
181+               6                0.333


**Verdict B: MIXED (weak).** Pages last updated 91–180 days ago under-capture slightly more often than pages updated in the last 30 days (41.6% vs 37.4%, n = 3,668 and 6,299). That's a gap of about 4 percentage points. The middle buckets are too small to read (31–90: n = 38; 181+: n = 6). Staleness is real context, but too weak to weight a CTR score. **What this changes in my rule:** staleness stays *out* of the score. It is shown as context in the top-10 review, so a weak signal doesn't push pages up the queue.

### The rule, in plain words

> A visible page (positions 1–20, ≥ 1,000 impressions in 90 days) is worth a title/meta review if its CTR is **below half** of what pages at the same narrow position earn. Its priority is the number of **clicks it is missing** against that expected CTR.

| Part | Value |
|---|---|
| Score | `missing_clicks = expected_ctr/100 × impressions_90d − clicks_90d`, only when `ctr < 0.5 × expected_ctr`; else 0 |
| Reason code (one) | `LOW_CTR_FOR_POSITION` |
| Action label | `REWRITE_TITLE_META` (flagged) / `NO_ACTION` (not flagged) |

In [3]:
RULE = {
    "population": "avg_position in (0, 20], impressions_90d >= 1000",
    "expected_ctr": "pooled CTR of the page's narrow position bucket (1-3, 3-5, 5-10, 10-15, 15-20)",
    "flag": "ctr < 0.5 * expected_ctr",
    "score": "missing clicks = expected_ctr/100 * impressions_90d - clicks_90d (0 if not flagged)",
    "reason_code": "LOW_CTR_FOR_POSITION",
    "action": "REWRITE_TITLE_META",
}
for k, v in RULE.items():
    print(f"{k:>12}: {v}")

  population: avg_position in (0, 20], impressions_90d >= 1000
expected_ctr: pooled CTR of the page's narrow position bucket (1-3, 3-5, 5-10, 10-15, 15-20)
        flag: ctr < 0.5 * expected_ctr
       score: missing clicks = expected_ctr/100 * impressions_90d - clicks_90d (0 if not flagged)
 reason_code: LOW_CTR_FOR_POSITION
      action: REWRITE_TITLE_META


## 2. Build the ranked queue (writes the CSV)

Inputs used: `avg_position`, `impressions_90d`, `clicks_90d`, `ctr`. All four are already observed at the decision moment. `content_id` is carried only so an editor can find the page. The CSV stays out of git (a CI rule) and this cell regenerates it on every run. The small metrics JSON is committed as the run's receipt.

In [4]:
q = vis.copy()
q["flagged"] = q["ctr"] < 0.5 * q["expected_ctr"]
q["missing_clicks"] = (q["expected_ctr"] / 100 * q["impressions_90d"] - q["clicks_90d"]).clip(lower=0)
q["score"] = np.where(q["flagged"], q["missing_clicks"], 0.0).round(1)
q["reason_code"] = np.where(q["flagged"], "LOW_CTR_FOR_POSITION", "")
q["action"] = np.where(q["flagged"], "REWRITE_TITLE_META", "NO_ACTION")
q = q.sort_values("score", ascending=False, kind="mergesort").reset_index(drop=True)
q["rank"] = np.arange(1, len(q) + 1)

OUT_COLS = ["rank", "content_id", "score", "reason_code", "action", "avg_position", "pos_bucket",
            "impressions_90d", "clicks_90d", "ctr", "expected_ctr"]
out_dir = root / "work/outputs"
out_dir.mkdir(parents=True, exist_ok=True)
q[OUT_COLS].to_csv(out_dir / "baseline_action_score.csv", index=False)

flagged = q[q["flagged"]]
top50_share = flagged["missing_clicks"].head(50).sum() / flagged["missing_clicks"].sum()
metrics = {
    "rule": RULE,
    "pages_in_population": int(len(q)),
    "pages_flagged": int(len(flagged)),
    "flag_rate": round(float(len(flagged) / len(q)), 4),
    "missing_clicks_total_flagged": round(float(flagged["missing_clicks"].sum()), 1),
    "share_of_missing_clicks_in_top50": round(float(top50_share), 4),
    "expected_ctr_by_bucket_pct": {k: round(float(v), 4) for k, v in bucket_ctr.items() if k in ["1-3", "3-5", "5-10", "10-15", "15-20"]},
    "signal_verdicts": {"ctr_vs_position": "MIXED", "staleness": "MIXED (weak)"},
}
(out_dir / "baseline_metrics.json").write_text(json.dumps(metrics, indent=2))

print(f"Wrote {out_dir.relative_to(root)}/baseline_action_score.csv ({len(q):,} rows) and baseline_metrics.json")
print(f"Flagged {len(flagged):,} of {len(q):,} visible pages ({len(flagged)/len(q):.1%})")
print(f"Top 50 hold {top50_share:.1%} of all missing clicks among flagged pages")

Wrote work/outputs/baseline_action_score.csv (10,011 rows) and baseline_metrics.json
Flagged 3,901 of 10,011 visible pages (39.0%)
Top 50 hold 21.7% of all missing clicks among flagged pages


## 3. Top-10 review

The top ten below, without IDs (clients relabeled A, B, C… for this view only). For each: the action, why it's there, and what would make it wrong.

In [5]:
top = q.head(10).copy()
letters = {cid: chr(65 + i) for i, cid in enumerate(dict.fromkeys(top["client_id"]))}
top["client"] = top["client_id"].map(letters)
VIEW = ["rank", "client", "avg_position", "impressions_90d", "clicks_90d", "ctr", "expected_ctr",
        "score", "main_intent", "freshness_tier", "engagement_rate", "sessions_90d", "action"]
top[VIEW].round(2)

,rank,client,avg_position,impressions_90d,clicks_90d,ctr,expected_ctr,score,main_intent,freshness_tier,engagement_rate,sessions_90d,action
0,1,A,2.5,509252,785,0.15,0.49,1716.6,informational,0-30,11.73,571,REWRITE_TITLE_META
1,2,A,4.2,517715,741,0.14,0.46,1620.3,informational,91-180,4.23,520,REWRITE_TITLE_META
2,3,B,2.3,272144,75,0.03,0.49,1261.9,informational,0-30,2.02,99,REWRITE_TITLE_META
3,4,C,4.7,213963,211,0.10,0.46,764.9,informational,0-30,0.86,116,REWRITE_TITLE_META
4,5,D,7.3,295097,154,0.05,0.29,708.0,informational,91-180,1.68,238,REWRITE_TITLE_META
5,6,A,2.9,149712,104,0.07,0.49,631.4,informational,0-30,5.94,101,REWRITE_TITLE_META
6,7,E,2.2,128068,17,0.01,0.49,612.1,informational,91-180,2.30,87,REWRITE_TITLE_META
7,8,D,9.7,208678,0,0.00,0.29,609.6,informational,91-180,0.00,6,REWRITE_TITLE_META
8,9,C,7.8,223271,70,0.03,0.29,582.2,informational,0-30,3.45,87,REWRITE_TITLE_META
9,10,C,7.6,140079,16,0.01,0.29,393.2,informational,0-30,0.00,8,REWRITE_TITLE_META


| # | Action | Why it's there | What would make it wrong |
|---|---|---|---|
| 1 | REWRITE_TITLE_META | Position 2.5, about 509K impressions, CTR 0.15% vs 0.49% expected. Missing the most clicks in the queue. | The query is broad and informational, and an AI Overview or featured snippet answers it on the results page. Then nobody clicks whatever the title says. |
| 2 | REWRITE_TITLE_META | Position 4.2, about 518K impressions, CTR 0.14% vs 0.46%. | Same client (A) and the same scale as #1. If both pages compete for the same queries, the fix is consolidation, not two rewrites. |
| 3 | REWRITE_TITLE_META | Position 2.3, about 272K impressions, only 75 clicks (0.03%). | A top-3 position with almost no clicks suggests the page ranks for queries it doesn't answer (query drift). Then it needs a content or intent fix, not a meta tweak. |
| 4 | REWRITE_TITLE_META | Position 4.7, about 214K impressions, CTR 0.10% vs 0.46%. | Engagement is only 0.86%, so the few visitors leave. If the snippet promises something the page doesn't deliver, a better title would only raise bounces. |
| 5 | REWRITE_TITLE_META | Position 7.3, about 295K impressions, CTR 0.05% vs 0.29%; not updated for 91–180 days. | A 90-day *average* of 7.3 can blend days at the top with days on page 2. If the position is unstable, the expected CTR is overstated. |
| 6 | REWRITE_TITLE_META | Position 2.9, about 150K impressions, CTR 0.07%. | This is the third page from client A in the top 6. A site-wide cause (brand results, missing rich results, a shared template) would need one site fix, not three page edits. |
| 7 | REWRITE_TITLE_META | Position 2.2, about 128K impressions, 17 clicks (0.01%). | A CTR this close to zero at the top of page 1 is suspicious. The impressions may come from another search surface (images, discover) or a tracking gap. Check Search Console's search appearance first. |
| 8 | REWRITE_TITLE_META | Position 9.7, about 209K impressions, **0 clicks**, 6 sessions. | **Weakest pick.** Zero clicks on 209K impressions doesn't look like a snippet problem; it looks like a data or query artifact. It also sits on the 10-position boundary, so it is barely in the 5–10 bucket. Verify the data before anyone edits. |
| 9 | REWRITE_TITLE_META | Position 7.8, about 223K impressions, CTR 0.03% vs 0.29%. | This is the second of client C's three pages in the top 10 (#4, #9, #10), all informational. If this client's whole topic sits under AI answers, rewriting titles won't move clicks. |
| 10 | REWRITE_TITLE_META | Position 7.6, about 140K impressions, 16 clicks, 0% engagement, 8 sessions. | Like #8, analytics is almost silent. If GA4 isn't connected properly for this page, "low CTR" may be partly a measurement gap.

## 4. Weak picks + leakage check

**Weak picks I found:**
- **#8 and #10 look like data problems, not snippet problems** (zero or near-zero clicks, single-digit sessions). The rule can't tell a broken measurement from a bad title. Week 5 idea: add a sanity gate such as "sessions consistent with clicks".
- **Client concentration.** Only five clients fill the top 10, and two of them hold three spots each (A: #1, #2, #6; C: #4, #9, #10). A per-client cap, or reviewing clients as a group, would spread editor time better.
- **All ten are informational pages.** The rule may systematically surface zero-click informational results. Expected CTR should probably also depend on intent, which the model can learn and a single rule can't.

**Leakage check:** the score uses only the four observed 90-day columns. The label-related columns (`trend_pct`, `trend_direction`) are not touched. No last-30 vs prev-30 split enters the score, and product flags don't exist in this data. The asserts below enforce that.

In [6]:
USED_INPUTS = {"avg_position", "impressions_90d", "clicks_90d", "ctr"}
FORBIDDEN = {"trend_pct", "trend_direction", "is_declining_label",
             "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"}
assert USED_INPUTS.isdisjoint(FORBIDDEN)
assert not any(c in RULE["score"] + RULE["flag"] for c in FORBIDDEN)
print("Score inputs:", sorted(USED_INPUTS))
print("Forbidden (label source / future window) - not used:", sorted(FORBIDDEN))

weak = top[(top["clicks_90d"] == 0) | (top["sessions_90d"] < 10)]
print(f"\nWeak picks flagged by a simple data-sanity check: ranks {weak['rank'].tolist()}")
print(f"Clients in top 10: {top['client'].nunique()} | max pages from one client: {top['client'].value_counts().max()}")
print(f"Intent mix of top 10: {top['main_intent'].value_counts().to_dict()}")

Score inputs: ['avg_position', 'clicks_90d', 'ctr', 'impressions_90d']
Forbidden (label source / future window) - not used: ['clicks_last_30d', 'impressions_last_30d', 'is_declining_label', 'sessions_last_30d', 'trend_direction', 'trend_pct']

Weak picks flagged by a simple data-sanity check: ranks [8, 10]
Clients in top 10: 5 | max pages from one client: 3
Intent mix of top 10: {'informational': 10}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.